In [1]:
import pandas as pd 
import numpy as np
import os

In [2]:
class Application_Normalizer:
    def __init__(self , path):
        self.df = pd.read_csv(path)
        self.dimension_tables = {}
        
    def normalize_column(self, column_name, save_dir='./data/normalized_data/'):
        unique_values = self.df[column_name].dropna().astype(str).str.lower().unique()
        dim_df = pd.DataFrame({
            f"{column_name}_id": range(1, len(unique_values) + 1),
            column_name: unique_values
        })
        
        self.dimension_tables[column_name] = dim_df

        # Merge and replace with ID
        self.df[column_name] = self.df[column_name].astype(str).str.lower()
        self.df = self.df.merge(dim_df, on=column_name, how='left')
        self.df.drop(columns=[column_name], inplace=True)

        # Save dimension table
        os.makedirs(save_dir, exist_ok=True)
        dim_df.to_csv(os.path.join(save_dir, f"{column_name}_table.csv"), index=False)

    def export_clean_application(self, output_path='./data/normalized_data/clean_applications.csv'):
        self.df.to_csv(output_path, index=False)

    
    def denormalize_and_clean(self):
        df_full = self.df.copy()
    
        for column_name, dim_df in self.dimension_tables.items():
            fk_col = f"{column_name}_id"
            
            if fk_col in df_full.columns:
                # Merge using the foreign key
                df_full = df_full.merge(dim_df, on=fk_col, how='left')
                
                # Drop the foreign key column
                df_full.drop(columns=[fk_col], inplace=True)
        
        return df_full

In [3]:
application_normalizer = Application_Normalizer('./data/cleaned_data/applications.csv')

In [4]:
columns_to_normalize = [
    'unique_id',
    'subscription_status',
    'voyage_role',
    'gender',
    'goal',
    'source',
    'country_name_from_country',
    'application_day_of_week'
]
for col in columns_to_normalize:
   application_normalizer.normalize_column(col) 

In [5]:
application_normalizer.export_clean_application()

In [6]:
cleaned_df = application_normalizer.denormalize_and_clean()

In [7]:
cleaned_df.to_csv('./data/normalized_data/merged_applications.csv' , index = False)